# Submission figures and tables

This notebook reads completed Compact and Extended summaries and exports the figures and plotted data to one final-results directory. It does not run allocation algorithms.

Each study exports mean error probability, a compact allocation-agreement table or extended two-panel allocation figure by $K$, and grouped mean-error bars by field family with 95% confidence intervals. Extended also exports a two-panel runtime line figure by $K$; Compact omits the runtime figure.

In [ ]:
# Resolve the project whether Jupyter starts here or from the repository root.
import importlib
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working = Path.cwd().resolve()
PROJECT = working / "problem-2-min-error-seq" if (working / "problem-2-min-error-seq").is_dir() else working
SOURCE = PROJECT / "src"
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))

from analysis import write_csv
_figures_module = importlib.reload(importlib.import_module("figures"))
load_case_rows = _figures_module.load_case_rows
load_summary = _figures_module.load_summary
save_figures = _figures_module.save_figures
summary_markdown = _figures_module.summary_markdown
allocation_match_table = _figures_module.allocation_match_table
allocation_match_markdown = _figures_module.allocation_match_markdown

## Result selection

Use `test` to build a fully reproducible draft from the completed test runs (90 compact and 180 extended cases per dimension). Change this value to `full` only after both full studies have completed successfully. Test and full exports are kept in separate directories.

In [ ]:
# Select the completed result scope used for every final table and figure.
RESULT_MODE = "test"
STUDIES = ("compact", "extended")
OUTPUT = PROJECT / "results" / "final" / RESULT_MODE

## Export matched summaries and figures

Each study retains its own matched sample. The exported tables contain the aggregate and case-level values used by the figures. Compact includes the Exhaustive reference, while Extended contains the four scalable methods.

In [ ]:
# Load, validate, export, and display each study in a fixed order.
for study in STUDIES:
    summary = load_summary(PROJECT, study, mode=RESULT_MODE)
    rows = load_case_rows(PROJECT, study, mode=RESULT_MODE)
    if not summary:
        raise RuntimeError(f"The {study} {RESULT_MODE} summary is empty")

    dimensions = {int(row["dimension"]) for row in summary}
    if dimensions != {1, 2}:
        raise RuntimeError(f"The {study} summary must contain both spatial dimensions")

    study_output = OUTPUT / study
    write_csv(study_output / "tables" / "error_summary.csv", summary)
    write_csv(study_output / "tables" / "case_errors.csv", rows)

    display(Markdown(f"## {study.title()} results"))
    display(Markdown(summary_markdown(summary)))

    if study == "compact":
        allocation_stats = allocation_match_table(rows)
        write_csv(
            study_output / "tables" / "allocation_match.csv",
            allocation_stats,
        )
        display(Markdown("### Allocation Match Statistics"))
        display(Markdown(allocation_match_markdown(allocation_stats)))

    figures = save_figures(
        summary,
        rows,
        study_output,
    )
    for name, figure in figures.items():
        display(Markdown(f"### {name.replace('_', ' ').title()}"))
        display(figure)
        plt.close(figure)

print("Final exports:", OUTPUT)